In [ ]:
# import tensorflow as tf

# from tensorflow.keras.applications import VGG16
# from tensorflow.keras.applications.vgg16 import preprocess_input

# # 1. Load training dataset
# train_data = tf.keras.utils.image_dataset_from_directory(
#     'seg_train/seg_train',
#     image_size=(224,224),
#     batch_size=32
# )

# # 2. Load testing dataset
# test_data = tf.keras.utils.image_dataset_from_directory(
#     'seg_test/seg_test',
#     image_size=(224,224),
#     batch_size=32
# )

# num_classes = len(train_data.class_names)

# # 3. Load pretrained VGG16
# base_model = VGG16(
#     weights='imagenet',
#     include_top=False,
#     input_shape=(224,224,3)
# )

# # Freeze layers
# base_model.trainable = False

# # 4. Create model
# model = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(224,224,3)),
#     tf.keras.layers.Lambda(preprocess_input),
#     base_model,
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(128, activation='relu'),
#     tf.keras.layers.Dense(num_classes, activation='softmax')
# ])

# # 5. Compile model
# model.compile(
#     optimizer='adam',
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# # 6. Train model
# history = model.fit(
#     train_data,
#     validation_data=test_data,
#     epochs=5
# )

# # 7. Evaluate model
# loss, accuracy = model.evaluate(test_data)
# print("Test Accuracy:", accuracy)

# # 8. Plot accuracy
# # plt.plot(history.history['accuracy'])
# # plt.plot(history.history['val_accuracy'])
# # plt.title('Model Accuracy')
# # plt.legend(['Training', 'Validation'])
# # plt.show()

# # # 9. Plot loss
# # plt.plot(history.history['loss'])
# # plt.plot(history.history['val_loss'])
# # plt.title('Model Loss')
# # plt.legend(['Training', 'Validation'])
# # plt.show()

# # 10. Fine tuning
# base_model.trainable = True

# for layer in base_model.layers[:-4]:
#     layer.trainable = False

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(0.0001),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# history2 = model.fit(
#     train_data,
#     validation_data=test_data,
#     epochs=3
# )

# # 11. Evaluate after fine tuning
# loss, accuracy = model.evaluate(test_data)
# print("Fine Tuned Accuracy:", accuracy)


In [1]:
import tensorflow as tf
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten,Dense,Dropout
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img,img_to_array
import matplotlib.pyplot as plt

dataset_path = r"D:\Downloads\LP IV Updated Datasets\LP IV Updated Datasets\Object Detection\Intel Image Classification"
weight_path = r"D:\Downloads\LP IV Updated Datasets\LP IV Updated Datasets\Object Detection\vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
classes = os.listdir(dataset_path)
X = []
y = []

for label,class_name in enumerate(classes):
    class_path = os.path.join(dataset_path,class_name)
    images = os.listdir(class_path)[:50]
    for image in images:
        image_path = os.path.join(class_path,image)
        try:
            img = load_img(image_path,target_size=(224,224))
            img = img_to_array(img)
            X.append(img)
            y.append(label)
        except:
            pass
X = np.array(X)
y = np.array(y)

X = X/255.0
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

base_model = VGG16(
    weights=None,
    include_top=False,
    input_shape=(224,224,3)
)
base_model.load_weights(weight_path)
base_model.trainable=False
model = Sequential([
    base_model,
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(len(classes),activation='softmax')
])

model.compile(
    optimizer = Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train,y_train,epochs=5,batch_size=8,validation_data=(X_test,y_test)
)

loss,accuracy = model.evaluate(X_test,y_test)
print("Loss before fine tuning : ",loss)
print("Accuracy before fine tuning : ",accuracy)

Epoch 1/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.3333 - loss: 2.4081 - val_accuracy: 0.6333 - val_loss: 0.9522
Epoch 2/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6458 - loss: 0.9470 - val_accuracy: 0.7000 - val_loss: 0.7522
Epoch 3/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.7208 - loss: 0.6664 - val_accuracy: 0.7500 - val_loss: 0.6866
Epoch 4/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.7792 - loss: 0.5154 - val_accuracy: 0.7167 - val_loss: 0.6831
Epoch 5/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.8208 - loss: 0.5100 - val_accuracy: 0.6833 - val_loss: 0.8900
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step - accuracy: 0.6833 - loss: 0.8900
Loss before fine tuning :  0.8900011777877808
Accuracy before fine tuning :  0.6833333373069763


In [2]:
# Unfreeze VGG16
base_model.trainable = True

# Freeze all layers except the last 4 layers
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Compile again with a small learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train again
model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=8,
    validation_data=(X_test, y_test)
)

loss, accuracy = model.evaluate(X_test, y_test)

print("\nFine Tuned Test Loss:", loss)
print("Fine Tuned Test Accuracy:", accuracy)

Epoch 1/2
30/30 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.8625 - loss: 0.3502 - val_accuracy: 0.7833 - val_loss: 0.5077
Epoch 2/2
30/30 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.8708 - loss: 0.3555 - val_accuracy: 0.7167 - val_loss: 0.6583
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step - accuracy: 0.7167 - loss: 0.6583

Fine Tuned Test Loss: 0.6583494544029236
Fine Tuned Test Accuracy: 0.7166666388511658
